In [ ]:
# Install Pytorch & other libraries
%pip install torch tensorboard

# Install Transformers
%pip install transformers
%pip install --upgrade transformers
# Install Hugging Face libraries
%pip install datasets accelerate evaluate bitsandbytes trl peft protobuf sentencepiece

# COMMENT IN: if you are running on a GPU that supports BF16 data type and flash attn, such as NVIDIA L4 or NVIDIA A100
#%pip install flash-attn

In [2]:
# Login into Hugging Face Hub
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Import the fine-tuning dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Meriem-DH/marine-dataset-cpt", split="train")
dataset = dataset.shuffle().select(range(419))

def format_cpt(sample):
    return {"text": sample["text"]}

dataset = dataset.map(format_cpt, remove_columns=dataset.column_names, batched=False)
dataset = dataset.train_test_split(test_size=0.2)
print(dataset["train"][0])

## Fine-tune Gemma using TRL and the SFTTrainer


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Hugging Face model id
model_id = "google/gemma-4-E2B-it" # @param ["google/gemma-4-E2B","google/gemma-4-E4B"] {"allow-input":true}

# Check if GPU benefits from bfloat16
if torch.cuda.get_device_capability()[0] >= 8:
    torch_dtype = torch.bfloat16
else:
    torch_dtype = torch.float16

# Define model init arguments
model_kwargs = dict(
    dtype=torch_dtype,
    device_map="auto", # Let torch decide how to load the model
)

# BitsAndBytesConfig: Enables 4-bit quantization to reduce model size/memory usage
model_kwargs["quantization_config"] = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=model_kwargs['dtype'],
    bnb_4bit_quant_storage=model_kwargs['dtype'],
)

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
tokenizer = AutoTokenizer.from_pretrained("google/gemma-4-E2B-it") # Load the Instruction Tokenizer to use the official Gemma template

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

The SFTTrainer has built-in support for PEFT, which allows you to fine-tune large language models efficiently without updating all their weights. Using QLoRA, you can combine model quantization with LoRA adapters to train massive models on limited GPU resources. To do this, you simply create a LoraConfig that specifies the adapter settings and pass it to the trainer, which will then automatically apply the adapters and train only those parameters, making the process much faster and memory-efficient.

In [5]:
from peft import LoraConfig

peft_config = LoraConfig(
    lora_alpha=16, # controls how much the LoRA adapters influence the original model weights
    lora_dropout=0.05, # during training, 5% of the adapter (A and B matrices) outputs are randomly zeroed out in each forward pass, helps regularize training to avoid overfitting
    r=8, # the rank of the low-rank decomposition in LoRA, it determines the size of the adapter matrices
    bias="none", # specifies whether to train biases in addition to LoRA adapters, the options are none or all or lora_only
    target_modules="all-linear", # defines which model layers the LoRA adapters are applied to
    task_type="CAUSAL_LM", # tells LoRA/PEFT the type of training task
    modules_to_save=["lm_head", "embed_tokens"], # Specifies additional modules to keep and save when saving the fine-tuned model
    ensure_weight_tying=True,
)

Before we start training, we need to define hyperparameter then we define the trainer object

In [6]:
import torch
from trl import SFTConfig

args = SFTConfig(
    output_dir="gemma-4-ocean-cpt",         # directory to save and repository id
    max_length=265,                         # max length for model and packing of the dataset
    num_train_epochs=1,                     # number of training epochs, set for 1 just for testing
    per_device_train_batch_size=1,          # batch size per device during training
    optim="adamw_torch_fused",
    logging_steps=10,                       # log every 10 steps
    save_strategy="epoch",                  # save checkpoint every epoch
    eval_strategy="epoch",                  # evaluate checkpoint every epoch
    learning_rate=5e-5,                     # learning rate
    fp16=False,   # half-precision 16-bit float, reduces memory and speeds up training, widely supported on GPUs
    bf16=True,   # 16-bit Brain float with larger numeric range but less fraction precision, only fully supported on certain GPUs (A100/H100)
    max_grad_norm=0.3,
    lr_scheduler_type="constant",           # use constant learning rate scheduler
    push_to_hub=True,                        # push model to hub
    hub_model_id="Meriem-DH/gemma-4-ocean-cpt",
    report_to="tensorboard",                # report metrics to tensorboard
    dataset_text_field="text",
    packing=True,
    dataset_kwargs={
        "add_special_tokens": False, # Template with special tokens
        "append_concat_token": True, # Add EOS token as separator token between examples
    }
)

In [ ]:
from trl import SFTTrainer

# Create Trainer object
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    processing_class=tokenizer,
)

In [ ]:
# Start training, the model will be automatically saved to the Hub and the output directory
trainer.train()

In [ ]:
# Optional : save the final model again to the Hugging Face Hub
trainer.save_model()

Before you can test your model, make sure to free the memory

In [ ]:
# free the memory again
del model
del trainer
torch.cuda.empty_cache()

## Test Model Inference



In [13]:
model_id

'google/gemma-4-E2B-it'

In [ ]:
from transformers import pipeline, GenerationConfig
from random import randint

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

config = GenerationConfig.from_pretrained(model_id)
config.max_new_tokens = 256

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

rand_idx = randint(0, len(dataset["test"]) - 1)
test_sample = dataset["test"][rand_idx]

prompt = test_sample["text"][:200]  # first 200 chars as prompt

outputs = pipe(prompt, generation_config=config)
print(f"Prompt:\n{prompt}")
print(f"Continuation:\n{outputs[0]['generated_text'][len(prompt):].strip()}")

In [ ]:
# save CPT adapter
trainer.save_model("gemma-4-ocean-cpt")
tokenizer.save_pretrained("gemma-4-ocean-cpt")

('ocean-AI-cpt/tokenizer_config.json',
 'ocean-AI-cpt/chat_template.jinja',
 'ocean-AI-cpt/tokenizer.json')

In [ ]:
trainer.push_to_hub("Meriem-DH/gemma-4-ocean-cpt")
tokenizer.push_to_hub("Meriem-DH/gemma-4-ocean-cpt")